# Homework Starter — Stage 05: Data Storage
Name: Ryan Mastropaolo
Date: 08/18/2026

Objectives:
- Env-driven paths to `data/raw/` and `data/processed/`
- Save CSV and Parquet; reload and validate
- Abstract IO with utility functions; document choices

In [1]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install numpy
# !pip install pandas
# !pip install pyarrow
# !pip install python-dotenv

In [2]:
# --- files this notebook needs (run me first - I only report, I change nothing) ---
from pathlib import Path

ROOT = Path.cwd()          # notebooks are meant to be run from their own folder
CHECKS = [
    (".env", "NEEDED", "YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing"),
    (".env.example", "NEEDED", "shipped with this stage - the template you copy to .env"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    print(f"\n{missing} needed file(s) missing. Put them at the paths above, relative to:\n  {ROOT}")
    print("If that folder looks wrong, you are running the notebook from the wrong place.")
else:
    print("\nAll needed files present.")

Looking in: /Users/ryanmastropaolo/bootcamp_ryan_mastropaolo/homework/homework05

  [OK ]  NEEDED    .env                                YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing
  [OK ]  NEEDED    .env.example                        shipped with this stage - the template you copy to .env

All needed files present.


In [3]:
import os, pathlib, datetime as dt
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
RAW = pathlib.Path(os.getenv('DATA_DIR_RAW', 'data/raw'))
PROC = pathlib.Path(os.getenv('DATA_DIR_PROCESSED', 'data/processed'))
RAW.mkdir(parents=True, exist_ok=True)
PROC.mkdir(parents=True, exist_ok=True)
print('RAW ->', RAW.resolve())
print('PROC ->', PROC.resolve())

RAW -> /Users/ryanmastropaolo/bootcamp_ryan_mastropaolo/homework/homework05/data/raw
PROC -> /Users/ryanmastropaolo/bootcamp_ryan_mastropaolo/homework/homework05/data/processed


## 1) Create or Load a Sample DataFrame
You may reuse data from prior stages or create a small synthetic dataset.

In [4]:
import numpy as np
dates = pd.date_range('2024-01-01', periods=20, freq='D')
df = pd.DataFrame({'date': dates, 'ticker': ['AAPL']*20, 'price': 150 + np.random.randn(20).cumsum()})
df.head()

,date,ticker,price
0,2024-01-01,AAPL,149.477079
1,2024-01-02,AAPL,149.176931
2,2024-01-03,AAPL,149.182366
3,2024-01-04,AAPL,148.644937
4,2024-01-05,AAPL,149.914585


## 2) Save CSV to data/raw/ and Parquet to data/processed/ (TODO)
- Use timestamped filenames.
- Handle missing Parquet engine gracefully.

In [5]:
def ts(): return dt.datetime.now().strftime('%Y%m%d-%H%M%S')

# TODO: Save CSV
csv_path = RAW / f"sample_{ts()}.csv"
df.to_csv(csv_path, index=False)
csv_path

# TODO: Save Parquet
pq_path = PROC / f"sample_{ts()}.parquet"
try:
    df.to_parquet(pq_path)
except Exception as e:
    print('Parquet engine not available. Install pyarrow or fastparquet to complete this step.')
    pq_path = None
pq_path

PosixPath('data/processed/sample_20260818-211041.parquet')

## 3) Reload and Validate (TODO)
- Compare shapes and key dtypes.

In [6]:
def validate_loaded(original, reloaded):
    checks = {
        'shape_equal': original.shape == reloaded.shape,
        'date_is_datetime': pd.api.types.is_datetime64_any_dtype(reloaded['date']) if 'date' in reloaded.columns else False,
        'price_is_numeric': pd.api.types.is_numeric_dtype(reloaded['price']) if 'price' in reloaded.columns else False,
    }
    return checks

df_csv = pd.read_csv(csv_path, parse_dates=['date'])
validate_loaded(df, df_csv)

{'shape_equal': True, 'date_is_datetime': True, 'price_is_numeric': True}

In [7]:
if pq_path:
    try:
        df_pq = pd.read_parquet(pq_path)
        pq_checks = validate_loaded(df, df_pq)
        print('Parquet validation:', pq_checks)
    except Exception as e:
        print('Parquet read failed:', e)

Parquet validation: {'shape_equal': True, 'date_is_datetime': True, 'price_is_numeric': True}


## 4) Utilities (TODO)
- Implement `detect_format`, `write_df`, `read_df`.
- Use suffix to route; create parent dirs if needed; friendly errors for Parquet.

In [8]:
import typing as t, pathlib

def detect_format(path: t.Union[str, pathlib.Path]):
    s = str(path).lower()
    if s.endswith('.csv'): return 'csv'
    if s.endswith('.parquet') or s.endswith('.pq') or s.endswith('.parq'): return 'parquet'
    raise ValueError('Unsupported format: ' + s)

def write_df(df: pd.DataFrame, path: t.Union[str, pathlib.Path]):
    p = pathlib.Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    fmt = detect_format(p)
    if fmt == 'csv':
        df.to_csv(p, index=False)
    else:
        try:
            df.to_parquet(p)
        except Exception as e:
            raise RuntimeError('Parquet engine not available. Install pyarrow or fastparquet.') from e
    return p

def read_df(path: t.Union[str, pathlib.Path]):
    p = pathlib.Path(path)
    fmt = detect_format(p)
    if fmt == 'csv':
        return pd.read_csv(p, parse_dates=['date']) if 'date' in pd.read_csv(p, nrows=0).columns else pd.read_csv(p)
    else:
        try:
            return pd.read_parquet(p)
        except Exception as e:
            raise RuntimeError('Parquet engine not available. Install pyarrow or fastparquet.') from e

# Demo: CSV round-trip via utilities
p_csv = RAW / f"util_{ts()}.csv"
write_df(df, p_csv)
df_csv_util = read_df(p_csv)
print('CSV util round-trip validation:', validate_loaded(df, df_csv_util))

# Demo: Parquet round-trip via utilities
p_pq = PROC / f"util_{ts()}.parquet"
try:
    write_df(df, p_pq)
    df_pq_util = read_df(p_pq)
    print('Parquet util round-trip validation:', validate_loaded(df, df_pq_util))
except RuntimeError as e:
    print('Skipping Parquet util demo:', e)

# Demo: write_df auto-creates missing directories
nested_path = RAW / 'nested_demo' / f"util_{ts()}.csv"
write_df(df, nested_path)
print('Auto-created missing directory:', nested_path.parent.exists(), '->', nested_path)

# Demo: unsupported suffix raises a clear error
try:
    detect_format('sample.txt')
except ValueError as e:
    print('Unsupported format correctly rejected:', e)

CSV util round-trip validation: {'shape_equal': True, 'date_is_datetime': True, 'price_is_numeric': True}
Parquet util round-trip validation: {'shape_equal': True, 'date_is_datetime': True, 'price_is_numeric': True}
Auto-created missing directory: True -> data/raw/nested_demo/util_20260818-211041.csv
Unsupported format correctly rejected: Unsupported format: sample.txt


## 5) Documentation
- Update README with a **Data Storage** section (folders, formats, env usage).
- Summarize validation checks and any assumptions.

**Summary:** See the `Data Storage` section of `README.md` in this folder for folder structure,
format choices, and how env variables (`DATA_DIR_RAW`, `DATA_DIR_PROCESSED`) are used to drive
`write_df`/`read_df`. Validation checks above confirmed: shapes match between original and
reloaded DataFrames for both CSV and Parquet, `date` reloads as `datetime64` from CSV (Parquet
preserves dtypes natively), and `price` stays numeric in both formats. `write_df`/`read_df`
route on file suffix, auto-create missing parent directories, and raise a clear `RuntimeError`
if no Parquet engine (pyarrow/fastparquet) is installed.